# ETL: Marts Gaucher (SIA-AM)

In [ ]:
import sys
from pathlib import Path
import duckdb

GAUCHER_PROCEDURES = (
    "0604240031",
    "0604240058",
    "0604240023",
    "0604630018",
)

ANO_INICIO = 2016
ANO_FIM = 2026

_FILTRO_COMPETENCIA = f"""
    length(TRIM(dt_atendimento)) = 6
    AND TRIM(dt_atendimento) ~ '^[0-9]{{6}}$'
    AND TRY_CAST(SUBSTR(dt_atendimento, 1, 4) AS INTEGER) BETWEEN {ANO_INICIO} AND {ANO_FIM}
    AND TRY_CAST(SUBSTR(dt_atendimento, 5, 2) AS INTEGER) BETWEEN 1 AND 12
    AND TRIM(dt_atendimento) NOT IN ('000000', '999999')
"""

_EXPR_DIAS_SOLIC_AUTORIZ = """
    CASE
        WHEN dt_solicitacao IS NULL OR dt_autorizacao IS NULL THEN NULL
        WHEN length(TRIM(dt_solicitacao)) <> 8 OR length(TRIM(dt_autorizacao)) <> 8
            THEN NULL
        WHEN TRIM(dt_solicitacao) IN ('00000000', '99999999')
            OR TRIM(dt_autorizacao) IN ('00000000', '99999999')
            THEN NULL
        ELSE date_diff(
            'day',
            TRY_CAST(try_strptime(TRIM(dt_solicitacao), '%Y%m%d') AS DATE),
            TRY_CAST(try_strptime(TRIM(dt_autorizacao), '%Y%m%d') AS DATE)
        )
    END
"""


def setup_views(con: duckdb.DuckDBPyConnection, parquet_glob: str) -> None:
    """Cria views raw, normalizada e Gaucher a partir dos parquets AM."""
    con.execute(
        f"""
        CREATE OR REPLACE VIEW tb_apac_medicamento_raw AS
        SELECT *
        FROM read_parquet('{parquet_glob}', union_by_name=true, filename=true);
        """
    )

    con.execute(
        """
        CREATE OR REPLACE VIEW tb_apac_medicamento AS
        SELECT
            AP_MVM AS dt_movimento_processamento,
            AP_CONDIC AS sg_tipo_gestao,
            AP_GESTAO AS cd_gestao_uf_municipio,
            AP_CODUNI AS cd_cnes_estabelecimento,
            AP_AUTORIZ AS cd_apac,
            AP_CMP AS dt_atendimento,
            AP_PRIPAL AS cd_procedimento_principal,
            TRY_CAST(AP_VL_AP AS DOUBLE) AS vl_apac_aprovada,
            AP_UFMUN AS cd_uf_municipio_estabelecimento,
            AP_TPUPS AS ct_tipo_estabelecimento,
            AP_TIPPRE AS ct_tipo_prestador,
            AP_MN_IND AS ct_estabelecimento_mantido_individual,
            AP_CNPJCPF AS cd_cnpj_estabelecimento,
            AP_CNPJMNT AS cd_cnpj_mantenedora,
            AP_CNSPCN AS cd_cns_paciente,
            AP_COIDADE AS cd_codigo_idade,
            TRY_CAST(AP_NUIDADE AS INTEGER) AS nu_idade_paciente,
            AP_SEXO AS sg_sexo_paciente,
            CASE AP_RACACOR
                WHEN '01' THEN 'Branca'
                WHEN '02' THEN 'Preta'
                WHEN '03' THEN 'Parda'
                WHEN '04' THEN 'Amarela'
                WHEN '05' THEN 'Indígena'
                WHEN '99' THEN 'Sem informação'
                ELSE NULL
            END AS ct_raca_cor_paciente,
            AP_MUNPCN AS cd_uf_municipio_residencia,
            AP_UFNACIO AS cd_nacionalidade_paciente,
            AP_CEPPCN AS cd_cep_paciente,
            CASE AP_UFDIF WHEN 1 THEN TRUE WHEN 0 THEN FALSE ELSE NULL END
                AS fl_uf_residencia_diferente,
            CASE AP_MNDIF WHEN 1 THEN TRUE WHEN 0 THEN FALSE ELSE NULL END
                AS fl_municipio_residencia_diferente,
            AP_DTINIC AS dt_inicio_validade,
            AP_DTFIM AS dt_fim_validade,
            AP_TPATEN AS ct_tipo_atendimento_apac,
            AP_TPAPAC AS ct_tipo_apac,
            AP_MOTSAI AS ct_motivo_saida_permanencia,
            CASE AP_OBITO WHEN '1' THEN TRUE WHEN '0' THEN FALSE ELSE NULL END AS fl_obito,
            CASE AP_ENCERR WHEN '1' THEN TRUE WHEN '0' THEN FALSE ELSE NULL END
                AS fl_encerramento,
            CASE AP_PERMAN WHEN '1' THEN TRUE WHEN '0' THEN FALSE ELSE NULL END
                AS fl_permanencia,
            CASE AP_ALTA WHEN '1' THEN TRUE WHEN '0' THEN FALSE ELSE NULL END AS fl_alta,
            CASE AP_TRANSF WHEN '1' THEN TRUE WHEN '0' THEN FALSE ELSE NULL END
                AS fl_transferencia,
            AP_DTOCOR AS dt_ocorrencia,
            AP_CODEMI AS cd_orgao_emissor,
            AP_CATEND AS ct_carater_atendimento,
            CASE AP_APACANT
                WHEN '000000000000' THEN 'aaa'
                ELSE AP_APACANT
            END AS cd_apac_anterior,
            AP_UNISOL AS cd_cnes_solicitante,
            AP_DTSOLIC AS dt_solicitacao,
            AP_DTAUT AS dt_autorizacao,
            AP_CIDPRI AS cd_cid_principal,
            CASE AP_CIDSEC WHEN '0000' THEN NULL ELSE AP_CIDSEC END AS cd_cid_secundario,
            CASE AP_CIDCAS WHEN '0000' THEN NULL ELSE AP_CIDCAS END AS cd_cid_causa_associada,
            AP_ETNIA AS ct_etnia_paciente,
            CAST(AM_PESO AS DOUBLE) AS nu_peso_kg,
            CAST(AM_ALTURA AS DOUBLE) AS nu_altura_cm,
            CASE AM_TRANSPL WHEN 'S' THEN TRUE WHEN 'N' THEN FALSE ELSE NULL END
                AS fl_pos_transplante,
            CAST(AM_QTDTRAN AS INTEGER) AS qt_transplantes,
            CASE AM_GESTANT WHEN 'S' THEN TRUE WHEN 'N' THEN FALSE ELSE NULL END AS fl_gestante
        FROM tb_apac_medicamento_raw
        QUALIFY row_number() OVER (
            PARTITION BY AP_AUTORIZ, AP_PRIPAL, AP_MVM
            ORDER BY AP_DTAUT DESC
        ) = 1;
        """
    )

    procs = ", ".join(f"'{p}'" for p in GAUCHER_PROCEDURES)
    con.execute(
        f"""
        CREATE OR REPLACE VIEW tb_apac_medicamento_gaucher AS
        WITH base AS (
            SELECT
                *,
                TRIM(cd_procedimento_principal) AS cd_proc,
                CASE TRIM(cd_procedimento_principal)
                    WHEN '0604240031' THEN 'Imiglucerase 400 U'
                    WHEN '0604240058' THEN 'Alfavelaglicerase 400 U'
                    WHEN '0604240023' THEN 'Alfataliglicerase 200 U'
                    WHEN '0604630018' THEN 'Miglustate 100 mg'
                END AS nm_medicamento,
                CASE TRIM(cd_procedimento_principal)
                    WHEN '0604630018' THEN 'ISS'
                    ELSE 'TRE'
                END AS ct_classe_terapeutica,
                TRY_CAST(nu_idade_paciente AS INTEGER) AS nu_idade_raw,
                TRY_CAST(nu_peso_kg AS DOUBLE) AS nu_peso_kg_num,
                TRY_CAST(nu_altura_cm AS DOUBLE) AS nu_altura_cm_num
            FROM tb_apac_medicamento
            WHERE TRIM(cd_procedimento_principal) IN ({procs})
        )
        SELECT
            b.*,
            CASE cd_codigo_idade
                WHEN '4' THEN nu_idade_raw
                WHEN '5' THEN nu_idade_raw
                WHEN '3' THEN nu_idade_raw / 12.0
                WHEN '2' THEN nu_idade_raw / 365.0
                ELSE nu_idade_raw
            END AS nu_idade_anos,
            CASE
                WHEN nu_peso_kg_num BETWEEN 1 AND 250 THEN nu_peso_kg_num
                ELSE NULL
            END AS nu_peso_kg_ok,
            CASE
                WHEN nu_altura_cm_num BETWEEN 30 AND 230 THEN nu_altura_cm_num
                ELSE NULL
            END AS nu_altura_cm_ok,
            CASE
                WHEN nu_peso_kg_num NOT BETWEEN 1 AND 250 THEN NULL
                WHEN nu_altura_cm_num NOT BETWEEN 30 AND 230 THEN NULL
                ELSE nu_peso_kg_num / power(nu_altura_cm_num / 100.0, 2)
            END AS nu_imc,
            SUBSTR(dt_atendimento, 1, 4) AS dt_ano_competencia,
            SUBSTR(dt_atendimento, 5, 2) AS dt_mes_competencia,
            TRY_CAST(try_strptime(TRIM(dt_atendimento) || '01', '%Y%m%d') AS DATE)
                AS dt_competencia_date,
            SUBSTR(cd_uf_municipio_estabelecimento, 1, 2) AS cd_uf_estabelecimento,
            SUBSTR(cd_uf_municipio_residencia, 1, 2) AS cd_uf_residencia,
            {_EXPR_DIAS_SOLIC_AUTORIZ} AS qt_dias_solic_autoriz,
            CASE TRIM(ct_tipo_apac)
                WHEN '1' THEN '1-Inicial'
                WHEN '2' THEN '2-Continuidade'
                ELSE 'Outro'
            END AS ct_tipo_apac_label
        FROM base b
        WHERE {_FILTRO_COMPETENCIA};
        """
    )


def build_marts(con: duckdb.DuckDBPyConnection, out_dir: Path) -> dict[str, Path]:
    """Agrega views Gaucher e grava parquets em out_dir."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    paths = {
        "mart_apac_mensal": out_dir / "mart_apac_mensal.parquet",
        "mart_paciente_mes": out_dir / "mart_paciente_mes.parquet",
        "mart_autorizacao": out_dir / "mart_autorizacao.parquet",
    }

    con.execute(
        f"""
        COPY (
            SELECT
                dt_atendimento AS dt_ano_mes,
                cd_uf_residencia,
                cd_uf_estabelecimento,
                nm_medicamento,
                ct_classe_terapeutica,
                ct_tipo_apac_label AS ct_tipo_apac,
                COUNT(*) AS qt_apacs,
                COUNT(DISTINCT cd_cns_paciente) AS qt_pacientes_distintos,
                SUM(CASE WHEN ct_tipo_apac_label = '1-Inicial' THEN 1 ELSE 0 END)
                    AS qt_iniciais,
                SUM(CASE WHEN ct_tipo_apac_label = '2-Continuidade' THEN 1 ELSE 0 END)
                    AS qt_continuidade
            FROM tb_apac_medicamento_gaucher
            WHERE cd_uf_residencia IS NOT NULL
            GROUP BY 1, 2, 3, 4, 5, 6
        ) TO '{paths["mart_apac_mensal"]}' (FORMAT PARQUET, COMPRESSION ZSTD);
        """
    )

    con.execute(
        f"""
        COPY (
            WITH paciente_mes AS (
                SELECT DISTINCT
                    SUBSTR(LOWER(hex(sha256(cd_cns_paciente))), 1, 10)
                        AS cd_paciente_hash,
                    dt_atendimento AS dt_ano_mes,
                    nm_medicamento,
                    ct_classe_terapeutica,
                    ROUND(nu_idade_anos, 1) AS nu_idade_anos,
                    sg_sexo_paciente AS sg_sexo,
                    ct_raca_cor_paciente AS ct_raca_cor,
                    cd_uf_residencia
                FROM tb_apac_medicamento_gaucher
                WHERE cd_cns_paciente IS NOT NULL
                  AND length(TRIM(cd_cns_paciente)) > 0
            ),
            com_lag AS (
                SELECT
                    *,
                    LAG(dt_ano_mes) OVER (
                        PARTITION BY cd_paciente_hash, nm_medicamento
                        ORDER BY dt_ano_mes
                    ) AS dt_ano_mes_anterior
                FROM paciente_mes
            )
            SELECT
                cd_paciente_hash,
                dt_ano_mes,
                nm_medicamento,
                ct_classe_terapeutica,
                nu_idade_anos,
                sg_sexo,
                ct_raca_cor,
                cd_uf_residencia,
                CASE
                    WHEN dt_ano_mes_anterior IS NULL THEN NULL
                    WHEN try_strptime(dt_ano_mes_anterior || '01', '%Y%m%d') IS NULL
                        OR try_strptime(dt_ano_mes || '01', '%Y%m%d') IS NULL
                        THEN NULL
                    ELSE CAST(
                        date_diff(
                            'month',
                            TRY_CAST(try_strptime(dt_ano_mes_anterior || '01', '%Y%m%d') AS DATE),
                            TRY_CAST(try_strptime(dt_ano_mes || '01', '%Y%m%d') AS DATE)
                        ) AS INTEGER
                    )
                END AS nu_gap_meses_desde_anterior
            FROM com_lag
        ) TO '{paths["mart_paciente_mes"]}' (FORMAT PARQUET, COMPRESSION ZSTD);
        """
    )

    con.execute(
        f"""
        COPY (
            SELECT
                dt_atendimento AS dt_ano_mes,
                cd_uf_residencia,
                nm_medicamento,
                quantile_cont(qt_dias_solic_autoriz, 0.5) AS qt_dias_p50,
                quantile_cont(qt_dias_solic_autoriz, 0.9) AS qt_dias_p90,
                COUNT(*) AS qt_registros
            FROM tb_apac_medicamento_gaucher
            WHERE qt_dias_solic_autoriz IS NOT NULL
              AND qt_dias_solic_autoriz BETWEEN 0 AND 365
              AND cd_uf_residencia IS NOT NULL
            GROUP BY 1, 2, 3
        ) TO '{paths["mart_autorizacao"]}' (FORMAT PARQUET, COMPRESSION ZSTD);
        """
    )

    return paths


def run_etl(base_dir: Path) -> dict[str, Path]:
    """Pipeline completo: raw AM -> marts processados."""
    base_dir = Path(base_dir)
    parquet_glob = str(base_dir / "data" / "raw" / "AM**" / "*.parquet")
    out_dir = base_dir / "data" / "processed"

    con = duckdb.connect(database=":memory:")
    setup_views(con, parquet_glob)
    return build_marts(con, out_dir)

In [ ]:
base_dir = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
_src = base_dir / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))


paths = run_etl(base_dir)

for name, p in paths.items():
    mb = p.stat().st_size / (1024 * 1024)
    print(f"{name}: {p} ({mb:.2f} MB)")